# 03. 추천 엔진 검증

**목적** — 문화 성향 · 고립·은둔 수준 · 지역 · 예산 조건을 바꿔가며 추천 결과가 설계 의도대로 달라지는지 확인

**추천 처리 과정** (`src/recommend_engine.py`)
1. 위치 · 거리 필터 (설문 Q17 응답 → 반경 km)
2. 예산 필터 (설문 Q18 응답 → 가격 상한, 가격 미확인 시설은 통과)
3. 활동 강도 필터 (은둔 ≤ 1 · 고립 ≤ 2 · 일반 ≤ 3, 결과가 없으면 1단계 완화)
4. 점수화 — 문화 성향 적합도(풀 안 +50 / 풀 밖 +10) · 프로그램 가중치(은둔 -30 / 고립 0 / 일반 +30)
   · 활동 강도 적합도 · 무거운 역사 콘텐츠 감점(은둔 -100 / 고립 -60) · 거리 감점(1km당 -0.5)
5. 카테고리 다양성 제한 (같은 묶음 최대 2건) → 최종 Top 5

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/데이터분석_포트폴리오/youth-culture-recommendation/notebooks"
!ls ../data | head
!pip install -q -r ../requirements.txt|
!apt-get -qq install fonts-nanum > /dev/null

Mounted at /content/drive
/content/drive/MyDrive/데이터분석_포트폴리오/youth-culture-recommendation/notebooks
raw
README.md
recommend_master_db.csv
survey_google.xlsx
survey_modu.xlsx
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [2]:
import sys
import pandas as pd

sys.path.append("../src")
from recommend_engine import recommend   # 추천 엔진 본체: src/recommend_engine.py

master = pd.read_csv("../data/recommend_master_db.csv", low_memory=False)   # 02번 노트북 결과물
print(f"마스터 DB: {len(master):,}건")

마스터 DB: 58,374건


## 1. 문화 성향 × 고립·은둔 수준 조합별 추천 (서울시청 기준)

In [3]:
test_personas = [
    {
        "name": "야외활동형+고립",
        "cluster": "야외활동형", "isolation_level": "고립",
        "distance_answer": "대중교통으로 30분 이내",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
    {
        "name": "문화예술형+일반",
        "cluster": "문화예술형", "isolation_level": "일반",
        "distance_answer": "거리는 상관없음",
        "price_answer": "1~3만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
    {
        "name": "야외활동형+은둔",
        "cluster": "야외활동형", "isolation_level": "은둔",
        "distance_answer": "무조건 집 근처! 걸어서 10~15분 이내",
        "price_answer": "무료 프로그램만 선호",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
    {
        "name": "집콕힐링형+일반",
        "cluster": "집콕힐링형", "isolation_level": "일반",
        "distance_answer": "대중교통으로 30분 이내",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
]

for p in test_personas:
    print(f"\n========== {p['name']} ==========")
    recs = recommend(master, p, top_n=3)
    print(recs[["facility_name", "mid_category", "is_program", "social_intensity_score", "match_reason"]])


========== 야외활동형+고립 ==========
               facility_name   mid_category  is_program  \
28593               남산&N서울타워            관광지       False   
28496  3.1운동 용산인쇄소 노동자 만세시위지            관광지       False   
55659              구 서울특별시청사  도보여행지(골목/스토리)       False   

       social_intensity_score                  match_reason  
28593                       2  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  
28496                       2  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  
55659                       1  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  

========== 문화예술형+일반 ==========
      facility_name mid_category  is_program  social_intensity_score  \
32986        온라인 접수        공연/행사        True                       2   
32954          서울광장        공연/행사        True                       2   
57047        507-14   문화예술교육프로그램        True                       2   

                                      match_reason  
32986  새로운 사람들과 어울리기 좋은 활동이에요. 사회적 연결을 넓혀보기에도 좋아요.  
32954  새로운 사람들과 어울리기 좋은 활동이에요. 사회적 연결을 넓혀보기에도 좋아요.  
570

In [4]:
test2_personas = [
    {
        "name": "집콕힐링형+고립",
        "cluster": "집콕힐링형", "isolation_level": "고립",
        "distance_answer": "대중교통으로 30분 이내",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
    {
        "name": "문화예술형+은둔",
        "cluster": "문화예술형", "isolation_level": "은둔",
        "distance_answer": "대중교통으로 30분 이내",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
    {
        "name": "문화예술형+고립",
        "cluster": "문화예술형", "isolation_level": "고립",
        "distance_answer": "대중교통으로 30분 이내",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    },
]

for p in test2_personas:
    print(f"\n========== {p['name']} ==========")
    recs = recommend(master, p, top_n=3)
    print(recs[["facility_name", "program_name", "mid_category", "is_program", "social_intensity_score", "match_reason"]])


========== 집콕힐링형+고립 ==========
      facility_name program_name mid_category  is_program  \
43893     응암1동 문화의집          NaN         문화의집       False   
43890  김중업 건축 문화의 집          NaN         문화의집       False   
35939   서울시청신청사다목적홀          NaN          전시관       False   

       social_intensity_score                  match_reason  
43893                       2  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  
43890                       2  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  
35939                       1  큰 부담 없이 가볍게 함께해볼 수 있는 활동이에요.  

========== 문화예술형+은둔 ==========
      facility_name program_name   mid_category  is_program  \
35939   서울시청신청사다목적홀          NaN            전시관       False   
40118      스마트서울전시관          NaN            전시관       False   
55659     구 서울특별시청사          NaN  도보여행지(골목/스토리)       False   

       social_intensity_score                match_reason  
35939                       1  혼자서도 부담 없이 둘러볼 수 있는 공간이에요.  
40118                       1  혼자서도 부담 없이 둘러볼 수 있는 공간이에요.  
55659     

## 2. 같은 조건에서 고립·은둔 수준만 바꿨을 때

In [5]:
# 거리/가격 고정, isolation_level만 바꿔서 순수 비교

for level in ["은둔", "고립", "일반"]:
    p = {
        "cluster": "야외활동형", "isolation_level": level,
        "distance_answer": "거리는 상관없음",
        "price_answer": "1만원 미만",
        "user_lat": 37.5665, "user_lon": 126.9780,
    }
    print(f"\n--- {level} ---")
    print(recommend(master, p, top_n=3)[["facility_name", "mid_category", "social_intensity_score", "is_program"]])


--- 은둔 ---
      facility_name   mid_category  social_intensity_score  is_program
55659     구 서울특별시청사  도보여행지(골목/스토리)                       1       False
51833          서울광장          국공립공원                       1       False
55456    서울 구 미국문화원  도보여행지(골목/스토리)                       1       False

--- 고립 ---
               facility_name   mid_category  social_intensity_score  \
28593               남산&N서울타워            관광지                       2   
28496  3.1운동 용산인쇄소 노동자 만세시위지            관광지                       2   
55659              구 서울특별시청사  도보여행지(골목/스토리)                       1   

       is_program  
28593       False  
28496       False  
55659       False  

--- 일반 ---
      facility_name mid_category  social_intensity_score  is_program
26991        전원일기마을      농어촌체험마을                       3        True
26829        천생연분마을      농어촌체험마을                       3        True
51854  성동 서울숲 여름캠핑장          캠핑장                       3       False


## 3. 지역별 추천 (대도시 · 중소도시 · 농촌)

대중교통 30분 이내 · 1만 원 미만 조건 (30분 이내 응답은 반경 10km로 환산)

In [6]:
test_locations = [
    {"name": "부산 시내", "lat": 35.1796, "lon": 129.0756},
    {"name": "전주 (중소도시)", "lat": 35.8242, "lon": 127.1480},
    {"name": "평창 (강원 산간)", "lat": 37.3702, "lon": 128.3900},
    {"name": "고흥 (전남 농촌)", "lat": 34.6111, "lon": 127.2858},
    {"name": "서산 (충남 외곽)", "lat": 36.7848, "lon": 126.4503},
]

for loc in test_locations:
    print(f"\n{'='*20} {loc['name']} {'='*20}")
    for level in ["은둔", "고립", "일반"]:
        p = {
            "cluster": "야외활동형", "isolation_level": level,
            "distance_answer": "대중교통으로 30분 이내",
            "price_answer": "1만원 미만",
            "user_lat": loc["lat"], "user_lon": loc["lon"],
        }
        recs = recommend(master, p, top_n=5)
        print(f"\n--- {level} ({len(recs)}건) ---")
        print(recs[["facility_name", "mid_category", "social_intensity_score", "is_program"]])


==================== 부산 시내 ====================

--- 은둔 (5건) ---
      facility_name   mid_category  social_intensity_score  is_program
15101          연화공원           도시공원                       1       False
43968          녹음광장          국공립공원                       1       False
55458  부산 박차정 의사 생가  도보여행지(골목/스토리)                       1       False
28170       박차정의사생가            관광지                       1       False
55562          영광도서  도보여행지(골목/스토리)                       1       False

--- 고립 (5건) ---
      facility_name   mid_category  social_intensity_score  is_program
28222           충렬사            관광지                       2       False
28171       장영실과학동산            관광지                       2       False
15101          연화공원           도시공원                       1       False
43968          녹음광장          국공립공원                       1       False
55458  부산 박차정 의사 생가  도보여행지(골목/스토리)                       1       False

--- 일반 (5건) ---
            facility_name mid_category  social_i

## 4. 경계 조건 확인

In [7]:
# 온라인 콘텐츠는 좌표가 없어도 가장 좁은 거리 조건에서 추천되어야 함
print(master[master["mid_category"] == "온라인콘텐츠"][["facility_name", "homepage", "social_intensity_score"]])

p = {"cluster": "집콕힐링형", "isolation_level": "은둔",
     "distance_answer": "무조건 집 근처! 걸어서 10~15분 이내",  # 일부러 가장 좁은 거리로
     "price_answer": "무료 프로그램만 선호",
     "user_lat": 37.5665, "user_lon": 126.9780}
print(recommend(master, p, top_n=5)[["facility_name","mid_category","social_intensity_score","match_reason"]])

               facility_name  \
57256          국립중앙도서관 전자도서관   
57257                국가전자도서관   
57258        KMDb 한국고전영화 VOD   
57259  Google Arts & Culture   
57260            국립현대미술관 온라인   
57261         국립중앙박물관 온라인전시관   
57262              사비나미술관 VR   
57263                   e뮤지엄   

                                                homepage  \
57256                               https://www.nl.go.kr   
57257                         https://www.dlibrary.go.kr   
57258                    https://www.kmdb.or.kr/vod/main   
57259                  https://artsandculture.google.com   
57260     https://www.mmca.go.kr/digitals/digitalMain.do   
57261                           https://www.museum.go.kr   
57262  https://www.savinamuseum.com/kor/exlist.action...   
57263                          https://www.emuseum.go.kr   

       social_intensity_score  
57256                       0  
57257                       0  
57258                       0  
57259                       0  
57260     

In [8]:
# 무료 선호인데 DIY키트가 안 나오는지 확인 (정확히 작동해야 함)
p1 = {"cluster": "집콕힐링형", "isolation_level": "은둔",
      "distance_answer": "거리는 상관없음", "price_answer": "무료 프로그램만 선호",
      "user_lat": 37.5665, "user_lon": 126.9780}
print(recommend(master, p1, top_n=10)[["facility_name","mid_category","price"]])
# DIY키트가 여기 있으면 안 됨

# 1만원 미만이면 DIY키트가 나오는지 확인
p2 = {**p1, "price_answer": "1만원 미만"}
print(recommend(master, p2, top_n=10)[["facility_name","mid_category","price"]])
# DIY키트가 여기는 있어야 함

       facility_name mid_category  price
57256  국립중앙도서관 전자도서관       온라인콘텐츠    0.0
57257        국가전자도서관       온라인콘텐츠    0.0
35939    서울시청신청사다목적홀          전시관    NaN
40118       스마트서울전시관          전시관    NaN
4955        한국금융사박물관          박물관    0.0
4536     석조전 대한제국역사관          박물관    NaN
4526   국립현대미술관(덕수궁관)          미술관    0.0
4678           일민미술관          미술관    0.0
24913      회현동 작은도서관          도서관    NaN
26400        우리소리도서관          도서관    NaN
             facility_name mid_category   price
57271  가죽공예(myleathertool)        DIY키트  5000.0
57261       국립중앙박물관 온라인전시관       온라인콘텐츠     0.0
57262            사비나미술관 VR       온라인콘텐츠     0.0
57265               101BOX        DIY키트  5000.0
35939          서울시청신청사다목적홀          전시관     NaN
40118             스마트서울전시관          전시관     NaN
4955              한국금융사박물관          박물관     0.0
4536           석조전 대한제국역사관          박물관     NaN
4526         국립현대미술관(덕수궁관)          미술관     0.0
4678                 일민미술관          미술관     0.0
